## YOLOV16

In [9]:
from ultralytics import YOLO
import time
import torch
import numpy as np

model_pt = YOLO("runs/detect/vehicle_detection/yolo26s_imbalanced/weights/best.pt")

# --- 1. Single-frame PyTorch latency ---
dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

# Warmup
for _ in range(10):
    model_pt(dummy, verbose=False)

torch.cuda.synchronize()
times = []
for _ in range(100):
    t0 = time.perf_counter()
    model_pt(dummy, verbose=False)
    torch.cuda.synchronize()
    times.append(time.perf_counter() - t0)

print(f"PyTorch single-frame: {np.median(times)*1000:.1f}ms median, "
      f"{np.percentile(times, 95)*1000:.1f}ms P95")

# --- 2. Export to TensorRT FP16 ---
# model_pt.export(format="engine", half=True, imgsz=640, dynamic=True)

# --- 3. TensorRT single-frame latency ---
model_trt = YOLO("runs/detect/vehicle_detection/yolo26s_imbalanced/weights/best.engine")

for _ in range(10):
    model_trt(dummy, verbose=False)

torch.cuda.synchronize()
times = []
for _ in range(100):
    t0 = time.perf_counter()
    model_trt(dummy, verbose=False)
    torch.cuda.synchronize()
    times.append(time.perf_counter() - t0)

print(f"TensorRT single-frame: {np.median(times)*1000:.1f}ms median, "
      f"{np.percentile(times, 95)*1000:.1f}ms P95")

# --- 4. Batched TensorRT latency (direct forward call)---
from ultralytics.nn.autobackend import AutoBackend

# Load the TensorRT engine directly
engine = AutoBackend(
    "runs/detect/vehicle_detection/yolo26s_imbalanced/weights/best.engine",
    device=torch.device("cuda:0"),
    fp16=True,
)

for batch_size in [1, 2, 4, 8]:
    batch_tensor = torch.zeros(batch_size, 3, 640, 640, dtype=torch.float16, device="cuda")

    # Warmup
    for _ in range(10):
        engine.forward(batch_tensor)
    torch.cuda.synchronize()

    times = []
    for _ in range(100):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        engine.forward(batch_tensor)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    total_ms = np.median(times) * 1000
    per_frame = total_ms / batch_size
    print(f"batch={batch_size}: {total_ms:.1f}ms total, "
          f"{per_frame:.1f}ms/frame, ~{1000/per_frame:.0f} FPS effective")

PyTorch single-frame: 9.7ms median, 12.4ms P95
Loading runs/detect/vehicle_detection/yolo26s_imbalanced/weights/best.engine for TensorRT inference...
TensorRT single-frame: 3.6ms median, 4.5ms P95
Loading runs/detect/vehicle_detection/yolo26s_imbalanced/weights/best.engine for TensorRT inference...
batch=1: 2.2ms total, 2.2ms/frame, ~447 FPS effective
batch=2: 2.2ms total, 1.1ms/frame, ~897 FPS effective
batch=4: 2.2ms total, 0.6ms/frame, ~1800 FPS effective
batch=8: 2.2ms total, 0.3ms/frame, ~3605 FPS effective


import cv2

# Simulate reading from RTSP/video stream
video_path = "video/sunset.mp4"
cap = cv2.VideoCapture(video_path)

times_full = []
for _ in range(100):
    ret, frame = cap.read()
    if not ret:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        ret, frame = cap.read()
    
    t0 = time.perf_counter()
    results = model_trt(frame, verbose=False)
    torch.cuda.synchronize()
    times_full.append(time.perf_counter() - t0)

cap.release()
print(f"Full pipeline (decode+resize+infer+NMS): "
      f"{np.median(times_full)*1000:.1f}ms median")

In [2]:
# ============================================================
# RF-DETR Medium Benchmark
# ============================================================

import time
import torch
import numpy as np
import cv2
from rfdetr import RFDETRMedium

# --- Load RF-DETR Medium (fine-tuned) ---
checkpoint_path = "output/checkpoint_best_total.pth"

model_rfdetr = RFDETRMedium(pretrain_weights="output/checkpoint_best_total.pth")
model_rfdetr.optimize_for_inference()

# Dummy image (numpy HWC, uint8 - same as YOLO benchmark)
dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

# ============================================================
# 1. PyTorch single-frame latency (RF-DETR)
# ============================================================
# Warmup
for _ in range(10):
    model_rfdetr.predict(dummy, threshold=0.5)

torch.cuda.synchronize()
times_rfdetr = []
for _ in range(100):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    model_rfdetr.predict(dummy, threshold=0.5)
    torch.cuda.synchronize()
    times_rfdetr.append(time.perf_counter() - t0)

print(f"RF-DETR-M PyTorch single-frame: {np.median(times_rfdetr)*1000:.1f}ms median, "
      f"{np.percentile(times_rfdetr, 95)*1000:.1f}ms P95")

# ============================================================
# 2. Full pipeline with video decode (RF-DETR)
# ============================================================
cap = cv2.VideoCapture("video/sunset.mp4")

times_full_rfdetr = []
for _ in range(100):
    ret, frame = cap.read()
    if not ret:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        ret, frame = cap.read()

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    model_rfdetr.predict(frame, threshold=0.5)
    torch.cuda.synchronize()
    times_full_rfdetr.append(time.perf_counter() - t0)

cap.release()
print(f"RF-DETR-M full pipeline: {np.median(times_full_rfdetr)*1000:.1f}ms median, "
      f"{np.percentile(times_full_rfdetr, 95)*1000:.1f}ms P95")

[2026-03-25 09:31:28] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-03-25 09:31:28] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-03-25 09:31:29] [INFO] rf-detr - Loading pretrain weights


[2026-03-25 09:31:29] [WARNING] rf-detr - Reinitializing detection head with 7 classes based on pretrained weights, configured for 90.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


RF-DETR-M PyTorch single-frame: 49.4ms median, 63.9ms P95
RF-DETR-M full pipeline: 61.4ms median, 90.3ms P95


In [3]:
# ============================================================
# Side-by-side comparison table
# ============================================================

# Use your YOLO26s numbers from earlier
yolo_pt_median = 9.7      # ms - update with your actual numbers
yolo_pt_p95 = 12.4
yolo_trt_median = 3.6
yolo_trt_p95 = 4.5
yolo_full_median = 4.4

rfdetr_pt_median = np.median(times_rfdetr) * 1000
rfdetr_pt_p95 = np.percentile(times_rfdetr, 95) * 1000
rfdetr_full_median = np.median(times_full_rfdetr) * 1000

print("=" * 70)
print(f"{'Benchmark':<35} {'YOLO26s':>15} {'RF-DETR-M':>15}")
print("=" * 70)
print(f"{'PyTorch single-frame (median)':<35} {yolo_pt_median:>12.1f}ms {rfdetr_pt_median:>12.1f}ms")
print(f"{'PyTorch single-frame (P95)':<35} {yolo_pt_p95:>12.1f}ms {rfdetr_pt_p95:>12.1f}ms")
print(f"{'TensorRT FP16 single-frame':<35} {yolo_trt_median:>12.1f}ms {'N/A':>15}")
print(f"{'Full pipeline (decode+infer)':<35} {yolo_full_median:>12.1f}ms {rfdetr_full_median:>12.1f}ms")
print(f"{'PyTorch FPS':<35} {1000/yolo_pt_median:>12.0f} {1000/rfdetr_pt_median:>12.0f}")
print(f"{'mAP@50 (test set)':<35} {'0.8681':>15} {'0.876':>15}")
print("=" * 70)

Benchmark                                   YOLO26s       RF-DETR-M
PyTorch single-frame (median)                9.7ms         49.4ms
PyTorch single-frame (P95)                  12.4ms         63.9ms
TensorRT FP16 single-frame                   3.6ms             N/A
Full pipeline (decode+infer)                 4.4ms         61.4ms
PyTorch FPS                                  103           20
mAP@50 (test set)                            0.8681           0.876
